<a href="https://colab.research.google.com/github/somustafa/plant_desease_detection/blob/main/tomato_disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

In [ ]:
from roboflow import Roboflow

API_KEY = "YOUR_API_KEY_HERE"
rf = Roboflow(api_key=API_KEY)

print('Dataset 1 yuklenir: Tomato Leaf Disease...')
project1 = rf.workspace("bryan-b56jm").project("tomato-leaf-disease-ssoha")
dataset1 = project1.version(63).download("yolov8")
print(f'Dataset 1 yuklendi: {dataset1.location}')

print('Dataset 2 yuklenir: Tomato Fruit Disease...')
project2 = rf.workspace("tomato-disease-detection-fcgsf").project("tomato-fruit-disease-detection")
dataset2 = project2.version(1).download("yolov8")
print(f'Dataset 2 yuklendi: {dataset2.location}')

In [ ]:
import yaml
from pathlib import Path

for name, ds in [('YARPAQ', dataset1), ('MEYVE', dataset2)]:
    with open(f'{ds.location}/data.yaml') as f:
        cfg = yaml.safe_load(f)
    print(f'\n{name} dataseti:')
    print(f'  Sinif sayi: {cfg["nc"]}')
    print(f'  Sinifler  : {cfg["names"]}')
    for split in ['train', 'valid', 'test']:
        d = Path(ds.location) / split / 'images'
        if d.exists():
            imgs = list(d.glob('*.*'))
            print(f'  {split:6s}: {len(imgs)} sekil')

In [ ]:
import os, shutil, yaml, random
from pathlib import Path

OUT = '/content/tomato_v3'
for split in ['train', 'valid', 'test']:
    os.makedirs(f'{OUT}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUT}/{split}/labels', exist_ok=True)

FINAL_CLASSES = [
    'bacterial_spot',
    'early_blight',
    'late_blight',
    'leaf_mold',
    'septoria_leaf_spot',
    'spider_mites',
    'yellow_leaf_curl_virus',
    'mosaic_virus',
    'healthy_leaf',
    'healthy_fruit',
    'anthracnose_fruit',
    'bacterial_spot_fruit',
    'blossom_end_rot',
    'spotted_wilt',
]
final_id = {name: i for i, name in enumerate(FINAL_CLASSES)}
print('Yekun sinifler:')
for i, name in enumerate(FINAL_CLASSES):
    grp = 'YARPAQ' if i < 9 else 'MEYVE '
    print(f'  {i:2d} [{grp}]: {name}')

In [ ]:
with open(f'{dataset1.location}/data.yaml') as f:
    cfg1 = yaml.safe_load(f)
print('Dataset 1 sinifler:', cfg1['names'])

KEYWORD_MAP = {
    'bacterial': 'bacterial_spot',
    'early':     'early_blight',
    'late':      'late_blight',
    'mold':      'leaf_mold',
    'septoria':  'septoria_leaf_spot',
    'spider':    'spider_mites',
    'mites':     'spider_mites',
    'yellow':    'yellow_leaf_curl_virus',
    'curl':      'yellow_leaf_curl_virus',
    'mosaic':    'mosaic_virus',
    'healthy':   'healthy_leaf',
}

cls_map1 = {}
print('\nDataset 1 mapping:')
for old_id, name in enumerate(cfg1['names']):
    norm = name.lower()
    matched = None
    for kw, cls in KEYWORD_MAP.items():
        if kw in norm:
            matched = cls
            break
    if matched:
        cls_map1[old_id] = final_id[matched]
        print(f'  {old_id}: {name} -> {matched}')
    else:
        print(f'  {old_id}: {name} -> SKIP')

In [ ]:
with open(f'{dataset2.location}/data.yaml') as f:
    cfg2 = yaml.safe_load(f)
print('Dataset 2 sinifler:', cfg2['names'])

FRUIT_KEYWORD_MAP = {
    'anthracnose' : 'anthracnose_fruit',
    'bacterial'   : 'bacterial_spot_fruit',
    'blossom'     : 'blossom_end_rot',
    'spotted_wilt': 'spotted_wilt',
    'wilt'        : 'spotted_wilt',
    'healthy'     : 'healthy_fruit',
}

cls_map2 = {}
print('\nDataset 2 mapping:')
for old_id, name in enumerate(cfg2['names']):
    norm = name.lower().replace(' ', '_').replace('-', '_')
    matched = None
    for kw, cls in FRUIT_KEYWORD_MAP.items():
        if kw in norm:
            matched = cls
            break
    if matched:
        cls_map2[old_id] = final_id[matched]
        print(f'  {old_id}: {name} -> {matched}')
    else:
        print(f'  {old_id}: {name} -> SKIP')

In [ ]:
MAX_PER_CLASS = 2000

def copy_dataset(ds_location, cls_map, out_dir, prefix):
    copied = 0
    for split in ['train', 'valid', 'test']:
        img_dir = Path(ds_location) / split / 'images'
        lbl_dir = Path(ds_location) / split / 'labels'
        if not img_dir.exists():
            continue
        for img_path in img_dir.glob('*.*'):
            if img_path.suffix.lower() not in ['.jpg','.jpeg','.png']:
                continue
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            new_lines = []
            for line in open(lbl_path):
                parts = line.strip().split()
                if not parts: continue
                old_id = int(parts[0])
                if old_id in cls_map:
                    new_lines.append(f'{cls_map[old_id]} {" ".join(parts[1:])}')
            if not new_lines: continue
            new_name = f'{prefix}_{img_path.name}'
            shutil.copy(img_path, f'{out_dir}/{split}/images/{new_name}')
            with open(f'{out_dir}/{split}/labels/{prefix}_{img_path.stem}.txt', 'w') as f:
                f.write('\n'.join(new_lines))
            copied += 1
    return copied

n1 = copy_dataset(dataset1.location, cls_map1, OUT, 'leaf')
n2 = copy_dataset(dataset2.location, cls_map2, OUT, 'fruit')

with open(f'{OUT}/data.yaml', 'w') as f:
    yaml.dump({
        'path': OUT, 'train': 'train/images',
        'val': 'valid/images', 'test': 'test/images',
        'nc': len(FINAL_CLASSES), 'names': FINAL_CLASSES
    }, f, default_flow_style=False)

print(f'Yarpaq sekillerinden kopyalandi: {n1}')
print(f'Meyve sekillerinden kopyalandi : {n2}')
print(f'Umumi: {n1+n2}')
for split in ['train', 'valid', 'test']:
    imgs = list(Path(f'{OUT}/{split}/images').glob('*.*'))
    print(f'  {split}: {len(imgs)}')

DATASET_PATH = OUT

In [ ]:
from pathlib import Path
import yaml

lbl_dir = Path(DATASET_PATH) / 'train' / 'labels'
sinif_sayi = {i: 0 for i in range(len(FINAL_CLASSES))}
for lbl in lbl_dir.glob('*.txt'):
    for line in open(lbl):
        parts = line.strip().split()
        if parts:
            sinif_sayi[int(parts[0])] += 1

print('Sinif uzre annotation sayi (train):')
for i, name in enumerate(FINAL_CLASSES):
    grp = 'YARPAQ' if i < 9 else 'MEYVE '
    bar = '|' * min(sinif_sayi[i] // 100, 50)
    print(f'  [{grp}] {name:30s}: {sinif_sayi[i]:6d} {bar}')

In [ ]:
import cv2, random
import matplotlib.pyplot as plt
from pathlib import Path

img_dir = Path(DATASET_PATH) / 'train' / 'images'
images = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.JPG')) + list(img_dir.glob('*.png'))

leaf_imgs  = [p for p in images if p.stem.startswith('leaf_')]
fruit_imgs = [p for p in images if p.stem.startswith('fruit_')]

samples = random.sample(leaf_imgs, min(4, len(leaf_imgs))) + \
          random.sample(fruit_imgs, min(4, len(fruit_imgs)))

COLORS = [(220,60,60),(60,180,60),(60,60,220),(220,180,0),(180,0,220),
          (0,180,180),(180,60,0),(60,0,180),(120,120,0),
          (0,200,100),(200,100,0),(100,0,200),(0,100,200),(200,0,100)]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = [ax for row in axes for ax in row]
fig.suptitle('Numune Sekiller — Yarpaq (yuxari) | Meyve (asagi)', fontsize=13, fontweight='bold')

for i, (ax, img_path) in enumerate(zip(axes, samples)):
    img = cv2.imread(str(img_path))
    if img is None: ax.axis('off'); continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in open(lbl_path):
            parts = line.strip().split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            cx,cy,bw,bh = map(float, parts[1:5])
            x1,y1 = int((cx-bw/2)*w), int((cy-bh/2)*h)
            x2,y2 = int((cx+bw/2)*w), int((cy+bh/2)*h)
            col = COLORS[cid % len(COLORS)]
            cv2.rectangle(img,(x1,y1),(x2,y2),col,2)
            lbl = FINAL_CLASSES[cid] if cid < len(FINAL_CLASSES) else str(cid)
            cv2.putText(img,lbl,(x1,max(y1-5,10)),cv2.FONT_HERSHEY_SIMPLEX,0.4,col,1)
    ax.imshow(img); ax.axis('off')
    ax.set_title(img_path.stem[:28], fontsize=7)

for ax in axes[len(samples):]: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print('Train baslayir — 14 sinif (9 yarpaq + 5 meyve)...')

results = model.train(
    data=f'{DATASET_PATH}/data.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project='/content/runs',
    name='tomato_v3',
    exist_ok=True,
    plots=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0, degrees=10.0,
)

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO

RUN = '/content/runs/tomato_v3'
best_model = YOLO(f'{RUN}/weights/best.pt')

display(Image(f'{RUN}/results.png', width=900))
display(Image(f'{RUN}/confusion_matrix_normalized.png', width=800))

metrics = best_model.val(data=f'{DATASET_PATH}/data.yaml', imgsz=640, device=0)
print('\nModel Performans:')
print(f'  mAP@50    : {metrics.box.map50:.3f}')
print(f'  mAP@50-95 : {metrics.box.map:.3f}')
print(f'  Precision : {metrics.box.mp:.3f}')
print(f'  Recall    : {metrics.box.mr:.3f}')
print('\n  Sinif uzre:')
for i, name in enumerate(FINAL_CLASSES):
    if i < len(metrics.box.ap):
        grp = 'YARPAQ' if i < 9 else 'MEYVE '
        print(f'  [{grp}] {name:30s}: {metrics.box.ap[i]:.3f}')

In [ ]:
import cv2, random
import matplotlib.pyplot as plt
from pathlib import Path

test_dir = Path(DATASET_PATH) / 'test' / 'images'
test_imgs = list(test_dir.glob('*.jpg')) + list(test_dir.glob('*.JPG')) + list(test_dir.glob('*.png'))

leaf_test  = [p for p in test_imgs if p.stem.startswith('leaf_')]
fruit_test = [p for p in test_imgs if p.stem.startswith('fruit_')]
samples = random.sample(leaf_test, min(4, len(leaf_test))) + \
          random.sample(fruit_test, min(4, len(fruit_test)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = [ax for row in axes for ax in row]
fig.suptitle('Test Neticeleri — Yarpaq | Meyve', fontsize=13, fontweight='bold')

for i, (ax, img_path) in enumerate(zip(axes, samples)):
    res = best_model.predict(source=str(img_path), conf=0.25, imgsz=640, device=0, verbose=False)[0]
    img = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.axis('off')
    if len(res.boxes) == 0:
        ax.set_title('Saglam', color='green', fontsize=9)
    else:
        dets = [f"{res.names[int(b.cls)][:14]}: {float(b.conf):.0%}" for b in res.boxes[:2]]
        ax.set_title('\n'.join(dets), color='red', fontsize=8)

for ax in axes[len(samples):]: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from google.colab import files
import cv2
import matplotlib.pyplot as plt

print('Pomidor sekli yukle (yarpaq ve ya meyve):')
uploaded = files.upload()

for fname in uploaded.keys():
    res = best_model.predict(source=fname, conf=0.25, imgsz=640, device=0, verbose=False)[0]
    img = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 7))
    plt.imshow(img); plt.axis('off')
    if len(res.boxes) == 0:
        plt.title('Saglam — Xestelik tapilmadi', color='green', fontsize=13)
    else:
        dets = [f"{res.names[int(b.cls)]}: {float(b.conf):.0%}" for b in res.boxes]
        plt.title('XESTELIK: ' + ' | '.join(dets), color='red', fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/Teknofest_VerticalFarm'
os.makedirs(save_dir, exist_ok=True)

shutil.copy('/content/runs/tomato_v3/weights/best.pt',
            f'{save_dir}/tomato_v3_best.pt')
print('Model saxlandi!')
print(f'{save_dir}/tomato_v3_best.pt')

In [ ]:
import json

LEAF_DISEASES  = FINAL_CLASSES[:8]
FRUIT_DISEASES = FINAL_CLASSES[10:]
HEALTHY        = ['healthy_leaf', 'healthy_fruit']

def analyze_tomato_frame(image_path, conf=0.3):
    """
    Robot kamerasinden gelen frame-i analiz edir.
    Hem yarpaq hem meyve xesteliklerini ayird edir.
    """
    res = best_model.predict(
        source=image_path, conf=conf,
        imgsz=640, device=0, verbose=False
    )[0]

    detections = []
    for box in res.boxes:
        cls_name   = res.names[int(box.cls)]
        confidence = float(box.conf)
        if cls_name in LEAF_DISEASES:
            dtype = 'leaf_disease'
        elif cls_name in FRUIT_DISEASES:
            dtype = 'fruit_disease'
        else:
            dtype = 'healthy'
        detections.append({
            'class': cls_name, 'type': dtype,
            'confidence': round(confidence, 3),
            'bbox': [round(x, 1) for x in box.xyxy[0].tolist()]
        })

    leaf_d  = [d for d in detections if d['type'] == 'leaf_disease']
    fruit_d = [d for d in detections if d['type'] == 'fruit_disease']

    if not leaf_d and not fruit_d:
        action = 'CONTINUE'
    elif any(d['confidence'] > 0.8 for d in leaf_d + fruit_d):
        action = 'ALERT'
    else:
        action = 'STOP_AND_FLAG'

    return {
        'healthy'            : len(leaf_d) == 0 and len(fruit_d) == 0,
        'leaf_disease_count' : len(leaf_d),
        'fruit_disease_count': len(fruit_d),
        'action'             : action,
        'detections'         : detections
    }

if test_imgs:
    result = analyze_tomato_frame(str(test_imgs[0]))
    print('Robot JSON netice:')
    print(json.dumps(result, indent=2, ensure_ascii=False))